Williams assumes a collisional cross section of sigma=1e-21 m^2 for all species. This needs to be checked in the `species.json` file being used for the simulations.

# setup

In [1]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

from pathlib import Path
from typing import Sequence

from IPython import get_ipython

import jax
import jax.numpy as jnp
import pandas as pd

import compressible.chemistry_types as chemistry_types
import compressible.chemistry_utils as chemistry_utils
import compressible.constants as constants
import compressible.energy_models_types as energy_models_types
from compressible.boundary_conditions_utils import build_boundary_arrays_1d_periodic
from compressible.equation_manager import run_scan
from compressible.equation_manager_types import EquationManager
from compressible.mesh import Mesh
from compressible.numerics_types import ClippingConfig, NumericsConfig
from compressible.state import compute_U_from_primitives, extract_primitives_from_U

import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"


def maybe_show_figure(fig: go.Figure) -> go.Figure:
    shell = get_ipython()
    if shell is not None and shell.__class__.__name__ == "ZMQInteractiveShell":
        fig.show()
    return fig


def maybe_write_image(fig: go.Figure, path: Path) -> None:
    try:
        import kaleido  # noqa: F401
    except ModuleNotFoundError:
        print(f"Skipping image export for {path.name}: kaleido is not installed.")
        return
    fig.write_image(str(path))

In [2]:
REPO_ROOT = Path("/home/hhoechter/tum/jaxfluids_internship")
DATA_DIR = REPO_ROOT / "data"
RESULTS_DIR = REPO_ROOT / "experiments" / "heatbath_0d_williams"

ENERGY_DATA_PATHS = {
    "bird": DATA_DIR / "air_5_bird_energy.json",
    "gnoffo": DATA_DIR / "air_5_gnoffo_equilibrium_enthalpy.json",
}
REACTION_DATA_PATHS = {
    "park": DATA_DIR / "park_reactions.json",
    "qk": DATA_DIR / "casseau_qk_reactions.json",
}
CHEMISTRY_MODEL_CONFIGS = {
    "park_pref": dict(model="park", park_vibrational_source="preferential_constant"),
    "park_nonpref": dict(model="park", park_vibrational_source="nonpreferential"),
    "cvdv_qp": dict(model="cvdv_qp"),
}
MODEL_DISPLAY_NAMES = {
    "park_pref": "Park preferential",
    "park_nonpref": "Park nonpreferential",
    "cvdv_qp": "CVDV-QP",
}
MODEL_COLORS = {
    "park_pref": "#1f77b4",
    "park_nonpref": "#d62728",
    "cvdv_qp": "#2ca02c",
}
REF_COLORS = ["#9467bd", "#ff7f0e", "#8c564b", "#e377c2", "#7f7f7f"]


def _extract_prim(U, equation_manager):
    prim = extract_primitives_from_U(U, equation_manager)
    return prim.Y_s, prim.rho, prim.T, prim.Tv, prim.p


extract_primitives_from_U_jitted = jax.jit(_extract_prim)


def normalize_rows(values: jnp.ndarray) -> jnp.ndarray:
    values = jnp.asarray(values)
    if values.ndim == 1:
        values = values[None, :]
    return values / jnp.clip(jnp.sum(values, axis=1, keepdims=True), 1e-14, None)


def load_species_table(
    species_names: Sequence[str],
    energy_model: str,
    include_electronic: bool,
) -> chemistry_utils.SpeciesTable:
    energy_cfg = energy_models_types.EnergyModelConfig(
        model=energy_model,
        include_electronic=include_electronic,
        data_path=str(ENERGY_DATA_PATHS[energy_model]),
    )
    return chemistry_utils.load_species_table(
        species_names=species_names,
        general_data_path=str(DATA_DIR / "air_5_gnoffo.json"),
        energy_model_config=energy_cfg,
    )


def build_time_controls(
    *,
    time_mode: str,
    dt: float | None = None,
    t_final: float,
    dt_fine: float | None = None,
    dt_coarse: float | None = None,
    t_threshold: float | None = None,
) -> tuple[float, float, jnp.ndarray | None]:
    if time_mode == "fixed":
        if dt is None:
            raise ValueError("time_mode='fixed' requires dt")
        return float(dt), float(t_final), None
    if time_mode == "two_phase":
        if dt_fine is None or dt_coarse is None or t_threshold is None:
            raise ValueError(
                "time_mode='two_phase' requires dt_fine, dt_coarse, and t_threshold"
            )
        n_fine = int(t_threshold / dt_fine)
        n_coarse = int((t_final - t_threshold) / dt_coarse)
        dt_array = jnp.concatenate(
            [
                jnp.full((n_fine,), dt_fine),
                jnp.full((n_coarse,), dt_coarse),
            ]
        )
        return float(dt_fine), float(t_final), dt_array
    raise ValueError(f"Unknown time_mode: {time_mode!r}")


def build_initial_state(
    *,
    equation_manager: EquationManager,
    species_names: Sequence[str],
    composition_basis: str,
    composition: dict[str, float],
    T_tr_init: float,
    T_V_init: float,
    p_init_atm: float,
) -> jnp.ndarray:
    species = equation_manager.species
    p_init_pa = p_init_atm * constants.ATM_TO_PA
    species_names = list(species_names)

    if composition_basis == "mole":
        total = sum(composition.get(name, 0.0) for name in species_names)
        Y_init = jnp.array(
            [[composition.get(name, 0.0) / total for name in species_names]]
        )
        rho_partial = {
            name: p_init_pa
            * composition.get(name, 0.0)
            / total
            * float(species.molar_masses[species.names.index(name)])
            / (constants.R_universal * T_tr_init)
            for name in species_names
        }
        rho_init = sum(rho_partial.values())
    elif composition_basis == "mass":
        Y_init = normalize_rows(
            jnp.array([composition[name] for name in species_names])
        )
        molar_masses = jnp.array(
            [species.molar_masses[species.names.index(name)] for name in species_names]
        )
        M_mix = jnp.sum(Y_init * molar_masses[None, :], axis=1)[0]
        rho_init = p_init_pa * float(M_mix) / (constants.R_universal * T_tr_init)
    else:
        raise ValueError(f"Unknown composition_basis: {composition_basis!r}")

    return compute_U_from_primitives(
        Y_s=Y_init,
        rho=jnp.array([rho_init]),
        u=jnp.array([0.0]),
        v=jnp.zeros(1),
        T_tr=jnp.array([T_tr_init]),
        T_V=jnp.array([T_V_init]),
        equation_manager=equation_manager,
    )


def run_williams_comparison(
    *,
    species_names: Sequence[str],
    composition_basis: str,
    composition: dict[str, float],
    T_tr_init: float,
    T_V_init: float,
    p_init_atm: float,
    energy_model: str,
    include_electronic: bool,
    comparison_models: Sequence[dict],
    dx: float,
    save_interval: int,
    time_mode: str,
    dt: float | None = None,
    t_final: float,
    dt_fine: float | None = None,
    dt_coarse: float | None = None,
    t_threshold: float | None = None,
) -> dict[str, dict]:
    dt0, t_final_value, dt_array = build_time_controls(
        time_mode=time_mode,
        dt=dt,
        t_final=t_final,
        dt_fine=dt_fine,
        dt_coarse=dt_coarse,
        t_threshold=t_threshold,
    )

    results = {}

    for model_spec in comparison_models:
        chemistry_model = model_spec["chemistry_model"]
        reaction_set = model_spec.get("reaction_set", "qk")
        model_energy = model_spec.get("energy_model", energy_model)
        model_include_electronic = model_spec.get(
            "include_electronic", include_electronic
        )

        species = load_species_table(
            species_names, model_energy, model_include_electronic
        )
        chemistry_model_config = chemistry_types.ChemistryModelConfig(
            **CHEMISTRY_MODEL_CONFIGS[chemistry_model]
        )
        reactions = chemistry_utils.load_reactions_from_json(
            json_path=str(REACTION_DATA_PATHS[reaction_set]),
            species_table=species,
            chemistry_model_config=chemistry_model_config,
        )
        included_reactions, excluded_reactions = (
            chemistry_utils.check_reaction_coverage(
                json_path=str(REACTION_DATA_PATHS[reaction_set]),
                species_names=species.names,
            )
        )

        mesh = Mesh.from_1d_grid(jnp.array([0.0, dx]), periodic=True)
        boundary_arrays = build_boundary_arrays_1d_periodic(mesh, species.n_species)
        numerics_config = NumericsConfig(
            dt=dt0,
            cfl=0.4,
            dt_mode="fixed",
            integrator_scheme="forward-euler",
            spatial_scheme="first_order",
            flux_scheme="hllc",
            clipping=ClippingConfig(),
        )
        equation_manager = EquationManager(
            species=species,
            reactions=reactions,
            numerics_config=numerics_config,
            boundary_arrays=boundary_arrays,
        )
        U_init = build_initial_state(
            equation_manager=equation_manager,
            species_names=species_names,
            composition_basis=composition_basis,
            composition=composition,
            T_tr_init=T_tr_init,
            T_V_init=T_V_init,
            p_init_atm=p_init_atm,
        )
        U_hist, t_hist = run_scan(
            U_init=U_init,
            mesh=mesh,
            equation_manager=equation_manager,
            t_final=t_final_value,
            save_interval=save_interval,
            dt_array=dt_array,
        )
        Y_s, rho, T, T_V, p = jax.vmap(
            extract_primitives_from_U_jitted, in_axes=(0, None)
        )(U_hist, equation_manager)
        mixture_molar_mass = jnp.sum(
            Y_s[:, 0, :] * species.molar_masses[None, :], axis=1
        )
        n_species_hist = (
            Y_s[:, 0, :] * rho / mixture_molar_mass[:, None] * constants.N_A
        )
        n_tot_init = (
            p_init_atm
            * constants.ATM_TO_PA
            / (constants.R_universal * T_tr_init)
            * constants.N_A
        )

        results[chemistry_model] = {
            "display_name": MODEL_DISPLAY_NAMES.get(chemistry_model, chemistry_model),
            "color": MODEL_COLORS.get(chemistry_model, "#000000"),
            "reaction_set": reaction_set,
            "energy_model": model_energy,
            "include_electronic": model_include_electronic,
            "equation_manager": equation_manager,
            "species": species,
            "U_hist": U_hist,
            "t": t_hist,
            "Y_s": Y_s,
            "rho": rho,
            "T": T,
            "T_V": T_V,
            "p": p,
            "n_species": n_species_hist,
            "n_tot_init": n_tot_init,
            "included_reactions": included_reactions,
            "excluded_reactions": excluded_reactions,
        }

        print(
            f"Completed {chemistry_model}: reaction_set={reaction_set}, energy_model={model_energy}"
        )

    return results


def print_comparison_summary(results: dict[str, dict]) -> None:
    for chemistry_model, result in results.items():
        print()
        print(f"=== {result['display_name']} ({chemistry_model}) ===")
        print("Included reactions:")
        for rxn in result["included_reactions"]:
            print(f"  {rxn['equation']}")
        print("Excluded reactions:")
        for rxn in result["excluded_reactions"]:
            print(f"  {rxn['equation']} - Missing: {list(rxn['missing_species'])}")


def load_reference_csv(csv_path: str | Path):
    df = pd.read_csv(csv_path, skiprows=1)
    with open(csv_path, "r", encoding="utf-8") as handle:
        dataset_names = [name for name in handle.readline().strip().split(",") if name]
    return df, dataset_names


def iter_reference_traces(df, dataset_names, suffixes):
    for i, name in enumerate(dataset_names):
        x_col = i * 2
        y_col = i * 2 + 1
        if y_col >= len(df.columns):
            continue
        x = pd.to_numeric(df.iloc[:, x_col], errors="coerce").dropna().values
        y = pd.to_numeric(df.iloc[:, y_col], errors="coerce").dropna().values
        n = min(len(x), len(y))
        prefix = name
        for suffix in suffixes:
            if suffix in name:
                prefix = name.replace(suffix, "")
                break
        yield prefix, name, x[:n], y[:n]


def make_temperature_comparison_plot(
    *,
    results: dict[str, dict],
    reference_csv: str | Path,
    title_prefix: str,
    yaxis_range: list[float],
    xaxis_range: list[float] | None = None,
    output_stem: str | None = None,
):
    df_ref, dataset_names_ref = load_reference_csv(reference_csv)

    for trace_key, fig_title, output_suffix in [
        ("_t_tr", f"{title_prefix} — T", "_t.pdf"),
        ("_t_v", f"{title_prefix} — T_V", "_v.pdf"),
    ]:
        fig = go.Figure()

        for chemistry_model, result in results.items():
            values = result["T"][:, 0] if trace_key == "_t_tr" else result["T_V"][:, 0]
            fig.add_trace(
                go.Scatter(
                    x=result["t"],
                    y=values,
                    mode="lines",
                    name=result["display_name"],
                    legendgroup=chemistry_model,
                    showlegend=True,
                    line=dict(
                        color=result["color"], shape="spline", smoothing=1.0, width=4
                    ),
                )
            )

        if trace_key == "_t_tr":
            fig.add_hline(
                y=7623.3, name="T_eq Casseau", line_dash="dash", line_color="black"
            )

        seen_prefixes = {}
        for prefix, name, x, y in iter_reference_traces(
            df_ref, dataset_names_ref, ["_t_v", "_t_tr"]
        ):
            if trace_key not in name:
                continue
            if prefix not in seen_prefixes:
                seen_prefixes[prefix] = REF_COLORS[len(seen_prefixes) % len(REF_COLORS)]
            color = seen_prefixes[prefix]
            fig.add_trace(
                go.Scatter(
                    x=x,
                    y=y * 1000,
                    mode="markers+lines",
                    name=prefix,
                    legendgroup=f"ref_{prefix}",
                    showlegend=prefix not in seen_prefixes,
                    line=dict(dash="dot", shape="spline", smoothing=1.0, color=color),
                    marker=dict(color=color, symbol="square", size=12),
                )
            )
            seen_prefixes[prefix] = color

        fig.update_xaxes(
            type="log", exponentformat="power", showexponent="all", showgrid=True
        )
        if xaxis_range is not None:
            fig.update_xaxes(range=xaxis_range)
        fig.update_yaxes(range=yaxis_range, showgrid=True)
        fig.update_layout(
            template="simple_white",
            title=fig_title,
            xaxis_title="Time (s)",
            yaxis_title="Temperature (K)",
            legend=dict(
                font=dict(size=18),
                x=0.5,
                y=-0.25,
                xanchor="center",
                yanchor="top",
                borderwidth=1,
                bordercolor="white",
                orientation="h",
            ),
            width=1200,
            height=800,
            margin=dict(t=10, b=100, l=80, r=10),
        )
    return fig


def make_species_comparison_plot(
    *,
    results: dict[str, dict],
    title: str,
    xaxis_range: list[float] | None = None,
):
    fig = go.Figure()
    palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]
    first_key = next(iter(results.keys()))
    species_names = results[first_key]["species"].names
    model_keys = list(results.keys())

    for model_idx, chemistry_model in enumerate(model_keys):
        result = results[chemistry_model]
        dash_style = (
            "solid" if model_idx == 0 else ("dot" if model_idx == 1 else "dash")
        )
        for species_idx, species_name in enumerate(species_names):
            fig.add_trace(
                go.Scatter(
                    x=result["t"],
                    y=result["n_species"][:, species_idx] / result["n_tot_init"],
                    mode="lines",
                    name=f"{result['display_name']} — {species_name}",
                    line=dict(
                        color=palette[species_idx % len(palette)],
                        dash=dash_style,
                        shape="spline",
                        smoothing=1.0,
                    ),
                )
            )

    fig.update_xaxes(
        type="log", exponentformat="power", showexponent="all", showgrid=True
    )
    if xaxis_range is not None:
        fig.update_xaxes(range=xaxis_range)
    fig.update_layout(
        template="simple_white",
        title=title,
        xaxis_title="Time (s)",
        yaxis_title="normalized number density",
        legend=dict(
            font=dict(size=18),
            x=0.5,
            y=-0.3,
            xanchor="center",
            yanchor="top",
            borderwidth=1,
            bordercolor="white",
            orientation="h",
        ),
        width=1200,
        height=800,
        margin=dict(t=10, b=120, l=80, r=10),
    )
    return fig

# air

In [3]:
print("=" * 80)
print("Williams Heatbath Comparison — Air")
print("=" * 80)

# --- user input start ---
species_names = ("N2", "N", "O2", "O", "NO")
composition_basis = "mole"
composition = {"N2": 0.767, "O2": 0.233, "N": 0.0, "O": 0.0, "NO": 0.0}

T_tr_init = 15000.0
T_V_init = 300.0
p_init_atm = 20.42

energy_model = "bird"
include_electronic = True
comparison_models = [
    {"chemistry_model": "park_pref", "reaction_set": "park"},
    {"chemistry_model": "park_nonpref", "reaction_set": "park"},
    # {"chemistry_model": "cvdv_qp", "reaction_set": "qk"},
]

# compare only a subset, e.g.
# comparison_models = [
#     {'chemistry_model': 'park_nonpref', 'reaction_set': 'park'},
#     {'chemistry_model': 'cvdv_qp', 'reaction_set': 'qk'},
# ]

dx = 1e-4
save_interval = 1

time_mode = "fixed"
dt = 1e-11
t_final = 1e-7
# --- user input end ---

air_results = run_williams_comparison(
    species_names=species_names,
    composition_basis=composition_basis,
    composition=composition,
    T_tr_init=T_tr_init,
    T_V_init=T_V_init,
    p_init_atm=p_init_atm,
    energy_model=energy_model,
    include_electronic=include_electronic,
    comparison_models=comparison_models,
    dx=dx,
    save_interval=save_interval,
    time_mode=time_mode,
    dt=dt,
    t_final=t_final,
)
print_comparison_summary(air_results)

Williams Heatbath Comparison — Air


E0412 23:23:11.002591    6379 cuda_executor.cc:1743] Could not get kernel mode driver version: ( INVALID_ARGUMENT: Version does not match the format X.Y.Z )
E0412 23:23:11.043685    6254 cuda_executor.cc:1743] Could not get kernel mode driver version: ( INVALID_ARGUMENT: Version does not match the format X.Y.Z )


Completed park_pref: reaction_set=park, energy_model=bird
Completed park_nonpref: reaction_set=park, energy_model=bird

=== Park preferential (park_pref) ===
Included reactions:
  O2 + N -> O + O + N
  O2 + NO -> O + O + NO
  O2 + N2 -> O + O + N2
  O2 + O2 -> O + O + O2
  O2 + O -> O + O + O
  N2 + O -> N + N + O
  N2 + O2 -> N + N + O2
  N2 + NO -> N + N + NO
  N2 + N2 -> N + N + N2
  N2 + N -> N + N + N
  NO + N2 -> N + O + N2
  NO + O2 -> N + O + O2
  NO + NO -> N + O + NO
  NO + O -> N + O + O
  NO + N -> N + O + N
  NO + O -> O2 + N
  N2 + O -> NO + N
  O2 + N -> NO + O
  NO + N -> N2 + O
Excluded reactions:

=== Park nonpreferential (park_nonpref) ===
Included reactions:
  O2 + N -> O + O + N
  O2 + NO -> O + O + NO
  O2 + N2 -> O + O + N2
  O2 + O2 -> O + O + O2
  O2 + O -> O + O + O
  N2 + O -> N + N + O
  N2 + O2 -> N + N + O2
  N2 + NO -> N + N + NO
  N2 + N2 -> N + N + N2
  N2 + N -> N + N + N
  NO + N2 -> N + O + N2
  NO + O2 -> N + O + O2
  NO + NO -> N + O + NO
  NO + O 

In [4]:
make_temperature_comparison_plot(
    results=air_results,
    reference_csv=RESULTS_DIR / "williams_figure_1_air.csv",
    title_prefix="Williams air comparison",
    yaxis_range=[0, 16000],
    output_stem="williams_air_comparison",
).show()

make_species_comparison_plot(
    results=air_results,
    title="Species number densities — Williams air comparison",
)

# pure N2

In [5]:
print("=" * 80)
print("Williams Heatbath Comparison — Pure N2")
print("=" * 80)

# --- user input start ---
species_names = ("N2", "N")
composition_basis = "mass"
composition = {"N2": 1.0, "N": 0.0}

T_tr_init = 20000.0
T_V_init = 300.0
p_init_atm = 27.25

energy_model = "bird"
include_electronic = True
comparison_models = [
    {"chemistry_model": "park_pref", "reaction_set": "park"},
    {"chemistry_model": "park_nonpref", "reaction_set": "park"},
    {"chemistry_model": "cvdv_qp", "reaction_set": "qk"},
]

dx = 1e-4
save_interval = 1

time_mode = "two_phase"
dt_fine = 1e-10
dt_coarse = 1e-8
t_threshold = 1e-7
t_final = 1e-6
# --- user input end ---

n2_results = run_williams_comparison(
    species_names=species_names,
    composition_basis=composition_basis,
    composition=composition,
    T_tr_init=T_tr_init,
    T_V_init=T_V_init,
    p_init_atm=p_init_atm,
    energy_model=energy_model,
    include_electronic=include_electronic,
    comparison_models=comparison_models,
    dx=dx,
    save_interval=save_interval,
    time_mode=time_mode,
    t_final=t_final,
    dt_fine=dt_fine,
    dt_coarse=dt_coarse,
    t_threshold=t_threshold,
)
print_comparison_summary(n2_results)

Williams Heatbath Comparison — Pure N2
Completed park_pref: reaction_set=park, energy_model=bird
Completed park_nonpref: reaction_set=park, energy_model=bird
Completed cvdv_qp: reaction_set=qk, energy_model=bird

=== Park preferential (park_pref) ===
Included reactions:
  N2 + N2 -> N + N + N2
  N2 + N -> N + N + N
Excluded reactions:
  O2 + N -> O + O + N - Missing: ['O', 'O2']
  O2 + NO -> O + O + NO - Missing: ['O', 'O2', 'NO']
  O2 + N2 -> O + O + N2 - Missing: ['O', 'O2']
  O2 + O2 -> O + O + O2 - Missing: ['O', 'O2']
  O2 + O -> O + O + O - Missing: ['O', 'O2']
  N2 + O -> N + N + O - Missing: ['O']
  N2 + O2 -> N + N + O2 - Missing: ['O2']
  N2 + NO -> N + N + NO - Missing: ['NO']
  NO + N2 -> N + O + N2 - Missing: ['O', 'NO']
  NO + O2 -> N + O + O2 - Missing: ['O', 'O2', 'NO']
  NO + NO -> N + O + NO - Missing: ['O', 'NO']
  NO + O -> N + O + O - Missing: ['O', 'NO']
  NO + N -> N + O + N - Missing: ['O', 'NO']
  NO + O -> O2 + N - Missing: ['O', 'O2', 'NO']
  N2 + O -> NO + N

In [6]:
make_temperature_comparison_plot(
    results=n2_results,
    reference_csv=RESULTS_DIR / "williams_figure_1_n2.csv",
    title_prefix="Williams pure N2 comparison",
    yaxis_range=[0, 20500],
    xaxis_range=[-10, -6],
    output_stem="williams_n2_comparison",
).show()

make_species_comparison_plot(
    results=n2_results,
    title="Species number densities — Williams pure N2 comparison",
    xaxis_range=[-10, -6],
)